# Notebook 04 — Language Family Clustering: Novel Technique 1

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

This notebook applies hierarchical clustering on the CKA similarity
matrix at each layer and tracks how language family groupings
**dissolve** as representations flow through the network.

## Research hypothesis

In early layers, languages from the same genetic family (e.g.,
Indo-European: English, Hindi, Bengali, Persian, German, French,
Spanish) should cluster together due to surface-level similarities.
In deeper layers, if Tiny Aya has learned language-agnostic
representations, these family-based clusters should dissolve.

## Metrics tracked

- **Cophenetic correlation**: How faithfully the dendrogram
  preserves pairwise CKA distances (quality of clustering).
- **Adjusted Rand Index (ARI)**: Agreement between discovered
  clusters and true family labels (1.0 = perfect match, 0 = random).
- **Family gap**: Intra-family CKA minus inter-family CKA.
  Convergence = gap approaching zero.

In [ ]:
import logging
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.languages import Language, LANGUAGE_FAMILIES
from src.analysis.cross_lingual_embedding_alignment.clustering import (
    compute_hierarchical_clustering,
    compute_family_dissolution_metrics,
)
from src.analysis.cross_lingual_embedding_alignment.visualization import (
    plot_dendrogram,
    plot_dendrograms_across_layers,
    plot_family_gap_curve,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

RESULTS_DIR = PROJECT_ROOT / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
CKA_DIR = RESULTS_DIR / "cka_matrices"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load CKA Matrices from Notebook 03

In [ ]:
# Load precomputed CKA matrices.
languages = list(Language)
language_names = [lang.lang_name for lang in languages]
iso_names = [lang.iso_code for lang in languages]

cka_matrices: dict[int, np.ndarray] = {}
for layer_idx in range(4):
    filepath = CKA_DIR / f"layer_{layer_idx}_linear_cka.npy"
    if filepath.exists():
        cka_matrices[layer_idx] = np.load(filepath)
    else:
        print(f"WARNING: {filepath} not found. Run Notebook 03 first.")

print(f"Loaded CKA matrices for {len(cka_matrices)} layers.")

## 2. Hierarchical Clustering at Each Layer

Apply Ward's method to the CKA distance matrices and compute
dendrograms. We expect dendrograms to become "flatter" (less
structured) in later layers as family distinctions dissolve.

In [ ]:
clustering_results: dict[int, dict] = {}

for layer_idx, matrix in sorted(cka_matrices.items()):
    result = compute_hierarchical_clustering(
        similarity_matrix=matrix,
        language_names=language_names,
        method="ward",
    )
    clustering_results[layer_idx] = result

    print(
        f"Layer {layer_idx}: cophenetic_corr = "
        f"{result['cophenetic_correlation']:.4f}"
    )

## 3. Side-by-Side Dendrograms

In [ ]:
fig = plot_dendrograms_across_layers(
    clustering_results=clustering_results,
    title="Language Cluster Dissolution Across Layers",
    save_path=str(FIGURES_DIR / "dendrograms_all_layers.png"),
)
plt.show()

# Individual dendrograms for higher resolution.
for layer_idx, result in sorted(clustering_results.items()):
    fig = plot_dendrogram(
        linkage_matrix=result["linkage_matrix"],
        language_names=[l.title() for l in language_names],
        layer_index=layer_idx,
        save_path=str(FIGURES_DIR / f"dendrogram_layer_{layer_idx}.png"),
    )
    plt.show()

## 4. Language Family Dissolution Metrics

For each layer, compute:
- Intra-family CKA (e.g., Hindi-Bengali, both Indo-European)
- Inter-family CKA (e.g., Hindi-Tamil, different families)
- Family gap = intra - inter
- ARI between discovered clusters and true families

In [ ]:
dissolution_results: dict[int, dict] = {}

print("\n=== Family Dissolution Metrics ===")
print(f"{'Layer':>5} {'Intra-Family':>14} {'Inter-Family':>14} "
      f"{'Gap':>8} {'ARI':>8} {'Cophenetic':>12}")
print("-" * 70)

for layer_idx, matrix in sorted(cka_matrices.items()):
    metrics = compute_family_dissolution_metrics(
        similarity_matrix=matrix,
        language_names=language_names,
        languages=languages,
    )
    dissolution_results[layer_idx] = metrics

    print(
        f"  {layer_idx:>3} {metrics['intra_family_cka']:>14.4f} "
        f"{metrics['inter_family_cka']:>14.4f} "
        f"{metrics['family_gap']:>8.4f} "
        f"{metrics['adjusted_rand_index']:>8.4f} "
        f"{metrics['cophenetic_correlation']:>12.4f}"
    )

## 5. Family Gap Convergence Plot

Plot intra-family vs inter-family CKA across layers. The gap
closing indicates that the model stops differentiating languages
by family.

In [ ]:
layer_indices = sorted(dissolution_results.keys())
intra_values = [dissolution_results[l]["intra_family_cka"] for l in layer_indices]
inter_values = [dissolution_results[l]["inter_family_cka"] for l in layer_indices]

fig = plot_family_gap_curve(
    layer_indices=list(layer_indices),
    intra_family_cka=intra_values,
    inter_family_cka=inter_values,
    save_path=str(FIGURES_DIR / "family_gap_curve.png"),
)
plt.show()

## 6. Cophenetic Correlation & ARI vs. Layer

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Cophenetic correlation.
coph_values = [dissolution_results[l]["cophenetic_correlation"] for l in layer_indices]
ax1.plot(list(layer_indices), coph_values, "o-", color="#2196F3", linewidth=2.5, markersize=8)
ax1.set_xlabel("Layer Index", fontsize=12)
ax1.set_ylabel("Cophenetic Correlation", fontsize=12)
ax1.set_title("Cophenetic Correlation vs. Layer", fontsize=13, fontweight="bold")
ax1.set_xticks(list(layer_indices))
ax1.grid(True, alpha=0.3)

# ARI.
ari_values = [dissolution_results[l]["adjusted_rand_index"] for l in layer_indices]
ax2.plot(list(layer_indices), ari_values, "s-", color="#FF9800", linewidth=2.5, markersize=8)
ax2.set_xlabel("Layer Index", fontsize=12)
ax2.set_ylabel("Adjusted Rand Index", fontsize=12)
ax2.set_title("ARI (Clusters vs. True Families) vs. Layer", fontsize=13, fontweight="bold")
ax2.set_xticks(list(layer_indices))
ax2.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "cophenetic_ari_vs_layer.png", bbox_inches="tight")
plt.show()

## 7. Per-Family Average CKA

In [ ]:
print("\n=== Per-Family Average Intra-CKA ===")
for layer_idx in layer_indices:
    per_family = dissolution_results[layer_idx]["per_family_avg_cka"]
    print(f"\nLayer {layer_idx}:")
    for family, avg_cka in sorted(per_family.items()):
        n_langs = len(LANGUAGE_FAMILIES.get(family, []))
        print(f"  {family:<20} avg_CKA={avg_cka:.4f} ({n_langs} languages)")

## 8. Save Metrics

In [ ]:
clustering_metrics: dict[str, float] = {}
for layer_idx, metrics in dissolution_results.items():
    prefix = f"layer_{layer_idx}"
    clustering_metrics[f"{prefix}_intra_family_cka"] = metrics["intra_family_cka"]
    clustering_metrics[f"{prefix}_inter_family_cka"] = metrics["inter_family_cka"]
    clustering_metrics[f"{prefix}_family_gap"] = metrics["family_gap"]
    clustering_metrics[f"{prefix}_adjusted_rand_index"] = metrics["adjusted_rand_index"]
    clustering_metrics[f"{prefix}_cophenetic_corr"] = metrics["cophenetic_correlation"]

with open(METRICS_DIR / "clustering_metrics.json", "w") as f:
    json.dump(clustering_metrics, f, indent=2)

print(f"\nMetrics saved to {METRICS_DIR / 'clustering_metrics.json'}")

## 9. Summary

**Key findings:**
- Dendrograms show [describe observed pattern] across layers.
- Family gap [increases/decreases] from layer 0 to layer 3.
- ARI [increases/decreases], indicating [interpretation].
- Next: Notebook 05 for anisotropy-corrected CKA analysis.